# CNNs: fine-tuning pre-trained CNNs into our dataset

In this lab, we will fine-tune a pre-trained CNN (ResNet-18) to perform classification of the dataset explored in the previous notebook.
Specifically, we will classify images of steel samples in one of the following classes: rolled-in scale (RS), patches (Pa), crazing (Cr), pitted surface (PS), inclusion (In) and scratches (Sc).

In [1]:
try:
    import google.colab
    IN_COLAB = True
    !git clone https://github.com/dskoda/ml4mat-26s-public.git
    !cd ml4mat-26s-public && pip install . && cd ..
    !pip install torch torchvision
    !pip install tqdm
    ROOT = "https://raw.githubusercontent.com/dskoda/ml4mat-26s-public/refs/heads/main/lectures/11-CNNs"
    STYLE = "colab"
except:
    IN_COLAB = False
    ROOT = "."
    STYLE = "jupyter"

In [2]:
import os
import io
import tqdm
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
from torchvision import transforms, datasets, models

import ml4mat_ucla as m4m

plt.style.use(STYLE)
m4m.utils.set_dpi(200)

## Loading the images

The commands below define a transform for the data, which will be preprocessed prior to be sent to the model.
Given that we are using a pre-trained model, we have to adapt to the standards of this model.
The transformations do the following:

- Convert the grayscale image to a 3-channel image (by repeating channels)
- Resize to 224x224 (the size expected by this ResNet model.)
- Convert to a tensor and normalize with ImageNet statistics.

If we were training a CNN from scratch, this would not be necessary

In [3]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # convert grayscale to 3 channels
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # ImageNet means
                         std=[0.229, 0.224, 0.225])    # ImageNet stds
])

Now, we can load the image from the dataset we downloaded, and split them into train, validation, and test sets.

In [4]:
# Create an ImageFolder dataset (labels are derived from folder names)
dataset = datasets.ImageFolder(root="data/NEU-DET/train/images", transform=transform)
print(f"Found {len(dataset)} images belonging to {len(dataset.classes)} classes: {dataset.classes}")

# Split dataset into training (80%) and validation (20%)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Create DataLoaders
batch_size = 256
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

test_dataset = datasets.ImageFolder(root="data/NEU-DET/validation/images", transform=transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

Found 1440 images belonging to 6 classes: ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']


## Loading the model

As before, we will load the model. However, differently from before, we will make a small "surgery" to the model to remove its last prediction layer (which was tailored to ImageNet) and convert it into something that predicts the classes of the images we are interested in looking at.
We will also add a softmax activation layer to assign a value of probability for each image.
In addition, because this is a simple fine-tuning process, we will freeze all convolution layers and train only the last NN layer.

In [5]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [6]:
# Load a pretrained ResNet18 model
model = models.resnet18(weights="DEFAULT")

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False
    
# Replace the final fully-connected layer to output six classes.
num_classes = len(dataset.classes)
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, num_classes),
    nn.Softmax(),
)

# Only the parameters of the final layer are trainable.
for param in model.fc.parameters():
    param.requires_grad = True
    
model = model.to(device)

# Define loss function and optimizer.
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=3e-4)

best_model = None
best_acc = 0

### Training the model

The process of fine-tuning the model is quite similar to the one of training a new model from scratch.
In essence, we implement below a training loop that minimizes the cross-entropy loss of the classification starting with the previous model.
The validation accuracy is used to store the best model and prevent overfitting.

In [7]:
num_epochs = 20  # adjust as needed

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    epoch_loss = running_loss / train_total
    epoch_acc = train_correct / train_total

    # Evaluate on the validation set
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total
    if val_acc > best_acc:
        best_acc = val_acc
        best_model = copy.deepcopy(model).cpu()

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Loss: {epoch_loss:.4f} "
          f"Train Acc: {epoch_acc:.4f} "
          f"Val Acc: {val_acc:.4f}")

print("Training complete.")

/opt/homebrew/Caskroom/miniconda/base/envs/ml4mat/lib/python3.10/site-packages/torch/nn/modules/module.py:1739: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


Epoch [1/20] Loss: 1.7783 Train Acc: 0.2144 Val Acc: 0.3333
Epoch [2/20] Loss: 1.7392 Train Acc: 0.3203 Val Acc: 0.3819
Epoch [3/20] Loss: 1.7014 Train Acc: 0.3863 Val Acc: 0.4201
Epoch [4/20] Loss: 1.6631 Train Acc: 0.5000 Val Acc: 0.4340
Epoch [5/20] Loss: 1.6240 Train Acc: 0.6076 Val Acc: 0.4792
Epoch [6/20] Loss: 1.5811 Train Acc: 0.6597 Val Acc: 0.5174
Epoch [7/20] Loss: 1.5416 Train Acc: 0.7161 Val Acc: 0.6389
Epoch [8/20] Loss: 1.5004 Train Acc: 0.7717 Val Acc: 0.7014
Epoch [9/20] Loss: 1.4627 Train Acc: 0.8082 Val Acc: 0.7604
Epoch [10/20] Loss: 1.4212 Train Acc: 0.8438 Val Acc: 0.8472
Epoch [11/20] Loss: 1.3890 Train Acc: 0.8889 Val Acc: 0.9167
Epoch [12/20] Loss: 1.3532 Train Acc: 0.9340 Val Acc: 0.9375
Epoch [13/20] Loss: 1.3253 Train Acc: 0.9470 Val Acc: 0.9583
Epoch [14/20] Loss: 1.2990 Train Acc: 0.9557 Val Acc: 0.9688
Epoch [15/20] Loss: 1.2800 Train Acc: 0.9644 Val Acc: 0.9653
Epoch [16/20] Loss: 1.2578 Train Acc: 0.9644 Val Acc: 0.9653
Epoch [17/20] Loss: 1.2435 Train 

### Testing the model performance

Finally, we will check the model performance on the held-out test set to see if everything looks good:

In [8]:
model = best_model
model.eval()
model = model.to(device)

val_correct = 0
val_total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        val_correct += (preds == labels).sum().item()
        val_total += labels.size(0)
final_val_acc = val_correct / val_total

print("Final Test Accuracy: {:.4f}".format(final_val_acc))

Final Test Accuracy: 0.9250
